# 12 — Accuracy Assessment

Calculates model accuracy metrics and observed-versus-predicted plots.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

data = pd.read_csv(PROCESSED_DIR / "station_samples" / "station_raster_samples.csv")
bundle = joblib.load(MODEL_DIR / "random_forest.joblib")
model = bundle["pipeline"]
features = bundle["features"]
target = bundle["target"]

X = data[features].replace([np.inf, -np.inf], np.nan)
y = data[target]
_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
pred = model.predict(X_test)

metrics = pd.DataFrame([{
    "rmse": mean_squared_error(y_test, pred) ** 0.5,
    "mae": mean_absolute_error(y_test, pred),
    "r2": r2_score(y_test, pred),
    "bias": float(np.mean(pred - y_test)),
}])

display(metrics)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, pred, alpha=0.6)
minimum = min(y_test.min(), pred.min())
maximum = max(y_test.max(), pred.max())
ax.plot([minimum, maximum], [minimum, maximum], linestyle="--")
ax.set_xlabel("Observed rainfall (mm)")
ax.set_ylabel("Predicted rainfall (mm)")
ax.set_title("Observed vs Predicted Rainfall")
plt.tight_layout()
plt.show()